# Can machine learning beat GARCH at forecasting risk?

**Question.** Notebook 02 showed that a classical statistical model (GARCH) forecasts next month's volatility better than simple rules. Can machine-learning models do better still, and is more model complexity worth it with five years of data?

**Approach**
- **One global model for all markets.** Instead of fitting each market separately, the ML models learn from all ten markets at once (a *panel*). That gives roughly ten times more training examples, which is the main advantage ML can have here.
- **Engineered features** known on day *t*: volatility over the last day, week, month, three months and all history; an exponentially weighted volatility; the size of recent falls (the "leverage effect"); recent returns; the US market's volatility and return; and whether the market is in Hong Kong.
- **Three learners, from simple to flexible:** ridge regression (regularised linear), gradient-boosted trees, and a small neural network.
- **Target:** log of next month's average squared return, converted back with a smearing correction so forecasts aren't biased low.
- **Leakage controls:** a walk-forward refit every quarter; *purging* (a training example counts only once its month-ahead window has finished); and US features lagged one day, because Hong Kong closes before New York opens. A unit test rewrites future prices and checks that no past forecast changes.

In [ ]:
import sqlite3
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("..") / "backend"))
import datavest_style  # site colours and fonts
from app import forecast as fc
from app import ml_forecast as ml
from app.config import WATCHLIST_GROUPS

warnings.filterwarnings("ignore", category=UserWarning)
NAMES = {s: n for g in WATCHLIST_GROUPS.values() for s, n in g.items()}
LABELS = {"last_month": "Last month", "long_run": "Long-run average", "ewma": "EWMA", "garch": "GARCH",
          "har": "HAR", "ridge": "Ridge (ML)", "gbm": "Gradient boosting (ML)", "mlp": "Neural network (ML)"}
pd.options.display.float_format = "{:,.3f}".format

with sqlite3.connect(Path("..") / "backend" / "datavest.db") as con:
    long = pd.read_sql("SELECT t.symbol, p.date, p.adj_close, p.close FROM prices p JOIN tickers t ON t.id = p.ticker_id",
                       con, parse_dates=["date"])
long["adj_close"] = long["adj_close"].fillna(long["close"])
prices = long.pivot(index="date", columns="symbol", values="adj_close").sort_index()
prices = prices[[s for s in NAMES if s in prices.columns]]
print(f"{prices.shape[1]} markets, {prices.index[0].date()} to {prices.index[-1].date()}")

## 1. The training data

One row per market per day. This is what the global model learns from.

In [ ]:
panel = ml.build_panel(prices)
print(f"{len(panel):,} rows ({panel['y'].notna().sum():,} with a known target), {len(ml.FEATURES)} features")
panel[["date", "symbol"] + ml.FEATURES[:6]].dropna().sample(5, random_state=0).set_index(["date", "symbol"])

## 2. Walk-forward forecasts: classical vs machine learning

All eight models are scored on exactly the same days.

In [ ]:
classical = {s: fc.walk_forward(prices[s].dropna(), min_train=500) for s in prices}
ml_fc = ml.walk_forward_ml(prices, min_train=500, refit_every=63)
merged = ml.combine_with_classical(classical, ml_fc)
qlike = pd.DataFrame({NAMES[s]: fc.evaluate(df)["qlike"] for s, df in merged.items()}).T
rel = qlike.div(qlike["last_month"], axis=0) - 1
print("forecast days per market:", min(len(d) for d in merged.values()), "to", max(len(d) for d in merged.values()))

In [ ]:
score = pd.DataFrame({
    "avg QLIKE vs last month": rel.mean(),
    "markets better than last month": qlike.lt(qlike["last_month"], axis=0).sum(),
    "markets better than GARCH": qlike.lt(qlike["garch"], axis=0).sum(),
}).rename(index=LABELS).sort_values("avg QLIKE vs last month")
score.style.format({"avg QLIKE vs last month": "{:+.0%}"}).bar(subset=["avg QLIKE vs last month"], align="zero", color=["#2a78d6", "#eb6834"])

In [ ]:
rel.rename(columns=LABELS).style.format("{:+.0%}").background_gradient(cmap="RdYlGn_r", vmin=-0.4, vmax=0.4).set_caption(
    "Error relative to the last-month baseline, by market (negative = better)")

## 3. Is more complexity worth it?

Models ordered from simplest to most flexible. If the most flexible models score worse than simpler ones, the extra flexibility is fitting noise instead of signal.

In [ ]:
order = ["last_month", "ewma", "garch", "har", "ridge", "gbm", "mlp"]
avg = rel.mean()[order]
fig, ax = plt.subplots(figsize=(10, 4))
colors = ["#5a6676" if m not in ml.ML_MODELS else "#2a78d6" for m in order]
ax.bar([LABELS[m].replace(" (ML)", "\n(ML)") for m in order], avg.values, color=colors, width=0.6)
ax.axhline(0, color="#1b2433", lw=1)
ax.yaxis.set_major_formatter(lambda y, _: f"{y:+.0%}")
ax.set_title("Average error vs last-month baseline (lower is better); blue = machine learning")
plt.xticks(fontsize=9);

## 4. What does the model look at?

**Permutation importance:** shuffle one input at a time on the most recent year and measure how much worse the gradient-boosting model gets. This explains the model without opening it up.

In [ ]:
imp = ml.feature_importance(prices).rename(index=ml.FEATURE_LABELS)
ax = imp.sort_values().plot.barh(color="#2a78d6", figsize=(9, 5))
ax.set_title("Permutation importance (drop in R² when the input is shuffled)"); ax.set_xlabel("");
ax.grid(axis="x", alpha=0.6); ax.grid(axis="y", visible=False);

## 5. Forecasts over time

In [ ]:
sym = "^HSI" if "^HSI" in merged else list(merged)[0]
f = merged[sym]
ax = f["target"].plot(color="#1b2433", lw=1.3, label="what happened")
f["garch"].plot(ax=ax, lw=1.1, label="GARCH", color="#2a78d6")
f["ridge"].plot(ax=ax, lw=1.1, label="Ridge (ML)", color="#1baf7a")
f["gbm"].plot(ax=ax, lw=1, label="Gradient boosting (ML)", color="#eb6834", alpha=0.85)
ax.yaxis.set_major_formatter(lambda y, _: f"{y:.0%}"); ax.set_title(f"{NAMES[sym]}: forecast vs realised volatility"); ax.legend(); ax.set_xlabel("");

## 6. Findings

In [ ]:
best = rel.mean().idxmin()
best_ml = rel.mean()[ml.ML_MODELS].idxmin()
lines = [
    f"Best model overall: **{LABELS[best]}** ({rel.mean()[best]:+.0%} error vs the last-month baseline).",
    f"Best machine-learning model: **{LABELS[best_ml]}** ({rel.mean()[best_ml]:+.0%}); GARCH scored {rel.mean()['garch']:+.0%}.",
    f"{LABELS[best_ml]} beat GARCH in {int(qlike[best_ml].lt(qlike['garch']).sum())} of {len(qlike)} markets.",
    f"Flexible models: gradient boosting {rel.mean()['gbm']:+.0%}, neural network {rel.mean()['mlp']:+.0%}. "
    + ("Both did worse than the linear model, a sign of overfitting with this little data."
       if min(rel.mean()['gbm'], rel.mean()['mlp']) > rel.mean()['ridge'] else "At least one beat the linear model."),
    f"Most important input: **{imp.index[0]}**.",
]
from IPython.display import Markdown
Markdown("\n\n".join("- " + l for l in lines))

## Limitations and next steps

- **Small data for ML.** About 12,000 market-days in total, and far fewer *independent* months. Flexible models need much more. Ten or twenty years of history, or many more markets, would give them a fairer chance.
- **Overlapping targets** make errors correlated. A Diebold–Mariano test with HAC errors would tell us whether the differences between the top models are statistically real.
- **Hyperparameters were chosen once, by judgement, not tuned on the test period.** Tuning on the test period would flatter the ML models. A proper approach is nested walk-forward tuning.
- **Next:** an LSTM on raw return sequences, which needs PyTorch and more data; features from option-implied volatility (VIX, VHSI); and quantile forecasts for risk ranges instead of single numbers.